# LongFlow — score the polish + combined night on GPU

Runtime: **any GPU**. Reads `polish_eval.zip` from Drive root. Scores:
(1) the knob curve (WER/sim per k), (2) combined-pool held-out, (3) the
four closed-loop renders vs the pre-registered bars (NOTES
"POLISH + COMBINED-POOL NIGHT PRE-REGISTRATION").


In [ ]:
# ===== COLD START — run me first, wait for READY =====
NOTEBOOK_VERSION = "Score polish+combined (GPU) v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU — pick a GPU runtime"
import glob, json, os, sys, zipfile
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1
from src.eval.metrics import clip_metrics, _ecapa, _normalizer, _whisper

from google.colab import drive
drive.mount("/content/drive")
candidates = glob.glob("/content/drive/MyDrive/polish_eval*.zip")  # root only, NOT recursive
assert candidates, "no polish_eval*.zip in Drive root"
zip_path = candidates[0]
print(f"using {zip_path} ({os.path.getsize(zip_path)/1e9:.2f} GB)")
AUD = "/content/eval_audio"
os.makedirs(AUD, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(AUD)
print(f"extracted {len(os.listdir(AUD))} entries")
print("READY")


In [ ]:
# ===== Knob curve + held-out + closed loop vs pre-registered bars =====
import jiwer
import torchaudio.functional as taf

with open(f"{AUD}/polish_report.json") as f:
    report = json.load(f)
with open(f"{AUD}/manifest.json") as f:
    manifest = json.load(f)
results = {"knob": {}, "held_out": {}, "closed_loop": {}, "verdicts": {}}

# ---- knob curve (teacher-forced, per k) ----
for utt_id, info in report.get("knob", {}).items():
    curve = {}
    for k, wav in sorted(info["k_wavs"].items(), key=lambda kv: int(kv[0])):
        m = clip_metrics(f"{AUD}/knob/{wav}", info["text"],
                         f"{AUD}/knob/{info['teacher']}", device="cuda")
        curve[k] = {"wer": round(m["wer"], 3), "sim": round(m["speaker_sim"], 3)}
        print(f"knob {utt_id} k={k}: wer={m['wer']:.3f} sim={m['speaker_sim']:.3f}")
    results["knob"][utt_id] = curve

# ---- combined held-out ----
entries = manifest["checkpoints"]["20000:M"]
rows = []
for e in entries:
    m = clip_metrics(f"{AUD}/{e['audio']}", e["text"],
                     f"{AUD}/{e['teacher_audio']}", device="cuda")
    rows.append({**m, "utt_id": e["utt_id"], "target_words": e["target_words"]})
wers = sorted(r["wer"] for r in rows)
sims = sorted(r["speaker_sim"] for r in rows)
by_bin = {}
for r in rows:
    by_bin.setdefault(r["target_words"], []).append(r["wer"])
results["held_out"]["20000:M"] = {
    "n": len(rows), "wer_median": wers[len(wers) // 2],
    "sim_median": sims[len(sims) // 2],
    "wer_by_bin": {b: sorted(v)[len(v) // 2] for b, v in sorted(by_bin.items())},
    "rows": rows,
}
o = results["held_out"]["20000:M"]
print(f"\n20000:M (combined pool): n={o['n']}  wer_med={o['wer_median']:.3f}  "
      f"sim_med={o['sim_median']:.3f}  by_bin={ {b: round(v,3) for b,v in o['wer_by_bin'].items()} }")
print("refs: cleanabl 0.123/0.914; mixed-B 0.163/0.901")

# ---- closed loop ----
WIN, HOP = 4.0, 2.0
ecapa = _ecapa("cuda")
n = _normalizer()
script_words = []
for line in report["cl_script"].splitlines():
    if ":" in line:
        line = line.split(":", 1)[1]
    script_words.append(line.strip())
SCRIPT = n(" ".join(w for w in script_words if w))
N_SCRIPT = len(SCRIPT.split())

def win_embs(path):
    x, sr = sf.read(path, dtype="float32")
    x16 = taf.resample(torch.from_numpy(x), sr, 16000).numpy()
    dur = len(x) / sr
    E, ts = [], []
    for i in range(int((dur - WIN) // HOP) + 1):
        seg = x16[int(i * HOP * 16000): int((i * HOP + WIN) * 16000)]
        if len(seg) < 16000:
            break
        emb = ecapa.encode_batch(torch.from_numpy(seg)[None].to("cuda"))[0, 0]
        E.append(emb.detach().cpu())
        ts.append(i * HOP)
    return torch.stack(E), np.array(ts), dur

def transcribe(path):
    segs, _ = _whisper("cuda").transcribe(str(path), language="en", beam_size=1)
    return " ".join((s.text or "").strip() for s in segs)

ref_E, _, _ = win_embs(f"{AUD}/t1_turnsplit_p0.wav")
ref = ref_E.median(0).values

def score(path):
    E, ts, dur = win_embs(path)
    sim = torch.nn.functional.cosine_similarity(E, ref[None], dim=-1).numpy()
    voice = sim >= 0.5
    horizon = 0.0
    for i in range(len(ts)):
        if voice[i]:
            horizon = ts[i] + WIN
    hyp = n(transcribe(path))
    nw = len(hyp.split())
    return {"duration_s": round(dur, 1),
            "wer_vs_script": round(jiwer.wer(SCRIPT, hyp) if hyp else 1.0, 3),
            "coverage_pct": round(100 * min(nw, N_SCRIPT) / N_SCRIPT, 1),
            "voice_pct": round(100 * float(voice.mean()), 1),
            "horizon_s": float(horizon),
            "sim_median": round(float(np.median(sim)), 3),
            "sim_final_third": round(float(np.median(sim[-max(1, len(sim) // 3):])), 3),
            "rate_wpm": round(60 * nw / dur, 1)}

for p in sorted(glob.glob(f"{AUD}/closed_loop/*.wav")):
    tag = os.path.basename(p)[:-4]
    results["closed_loop"][tag] = score(p)
    print(f"{tag}: {results['closed_loop'][tag]}", flush=True)

# ---- pre-registered verdicts ----
c = results["closed_loop"]
CLEANABL = {"wer": (0.116, 0.158), "sim": (0.474, 0.572)}
merged = [t for t, r in c.items()
          if r["wer_vs_script"] <= 0.07 and r["sim_median"] >= 0.50]
if merged:
    results["verdicts"]["merged_success"] = (
        f"MERGED SUCCESS candidates (WER<=0.07, sim>=0.50): {merged} — ear confirms or denies")
comb = [c.get("combined_cfg_heun8_s0"), c.get("combined_cfg_heun8_s1")]
if all(comb):
    ok = all(r["wer_vs_script"] <= 0.09 and r["sim_median"] >= 0.45 for r in comb)
    o = results["held_out"]["20000:M"]
    results["verdicts"]["combined"] = (
        ("COMBINED WINS — becomes the offline base" if ok and o["wer_median"] <= 0.123
         else "combined did not clear the pre-registered bars — read rows before deciding"))
for tag in ("cleanabl_polish2_s0", "combined_polish2_s0"):
    if tag in c:
        r = c[tag]
        base = CLEANABL if tag.startswith("cleanabl") else None
        results["verdicts"][tag] = (
            f"polish k=2 loop: WER {r['wer_vs_script']}, sim {r['sim_median']}, "
            f"voice {r['voice_pct']}% — compare vs its no-polish twin; ear decides listenability")

print("\nVERDICTS:", json.dumps(results["verdicts"], indent=2))
with open("/content/drive/MyDrive/polish_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("metrics on Drive root: polish_metrics.json")
print("\nListening (never skipped): the knob series k0->k5, then the two polish loop renders.")
